# If you want to access the version you have already modified, click "Edit"
# If you want to access the original sample code, click "...", then click "Copy & Edit Notebook"

In [1]:
## This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        pass
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# Import necessary packages.
import numpy as np
import torch
import os
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
# "ConcatDataset" and "Subset" are possibly useful when doing semi-supervised learning.
from torch.utils.data import ConcatDataset, DataLoader, Subset, Dataset
from torchvision.datasets import DatasetFolder, VisionDataset

# This is for the progress bar.
from tqdm.auto import tqdm
import random

d:\anaconda\envs\bupt\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\anaconda\envs\bupt\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: Could not find module 'D:\anaconda\envs\bupt\Lib\site-packages\torchvision\image.pyd' (or one of its dependencies). Try using the full path with constructor syntax.
  warn(f"Failed to load image Python extension: {e}")


In [4]:
myseed = 6666  # set a random seed for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(myseed)
torch.manual_seed(myseed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(myseed)

## **Transforms**
Torchvision provides lots of useful utilities for image preprocessing, data wrapping as well as data augmentation.

Please refer to PyTorch official website for details about different transforms.

In [36]:
# Normally, We don't need augmentations in testing and validation.
# All we need here is to resize the PIL image and transform it into Tensor.

class RandomGrayscale:
    def __init__(self, p=0.5, num_output_channels=3):
        self.p = p
        self.num_output_channels = num_output_channels

    def __call__(self, img):
        if random.random() < self.p:
            return transforms.functional.rgb_to_grayscale(img, num_output_channels=self.num_output_channels)
        return img
    
class RandomGaussianBlur:
    def __init__(self, p=0.5, kernel_size=3):
        self.p = p
        self.kernel_size = kernel_size

    def __call__(self, img):
        if random.random() < self.p:
            return transforms.functional.gaussian_blur(img, kernel_size=self.kernel_size)
        return img

test_tfm = transforms.Compose([
    transforms.Resize(128),
    transforms.RandomCrop(128),  # 随机裁剪
    transforms.RandomHorizontalFlip(p=0.5),  # 随机水平翻转
    transforms.RandomVerticalFlip(p=0.5),    # 随机垂直翻转
#     transforms.RandomAffine(degrees=(30, 70), translate=(0.1, 0.3), scale=(0.5, 0.75)),
#     transforms.ColorJitter(brightness=0.6, contrast = 0.3, saturation=0.2, hue = 0.1), # 颜色抖动
    transforms.RandomRotation(degrees=60),  # 随机旋转
    RandomGaussianBlur(p = 0.5 , kernel_size = 3),  # 高斯模糊
    RandomGrayscale(p = 0.1 , num_output_channels = 3),  # 转为灰度图
    transforms.ToTensor()
])

# However, it is also possible to use augmentation in the testing phase.
# You may use train_tfm to produce a variety of images and then test using ensemble methods

train_tfm = transforms.Compose([
    transforms.Resize(128),
    transforms.RandomCrop(128),  # 随机裁剪
    transforms.RandomHorizontalFlip(p=0.5),  # 随机水平翻转
    transforms.RandomVerticalFlip(p=0.5),    # 随机垂直翻转
#     transforms.RandomAffine(degrees=(30, 70), translate=(0.1, 0.3), scale=(0.9, 1)),
    transforms.ColorJitter(brightness=0.6, contrast = 0.3, saturation=0.2, hue = 0.1), # 颜色抖动
    transforms.RandomRotation(degrees=60),  # 随机旋转
    RandomGaussianBlur(p = 0.5 , kernel_size = 3),  # 高斯模糊
    RandomGrayscale(p = 0.1 , num_output_channels = 3),  # 转为灰度图
    transforms.ToTensor()
    # transforms.Lambda(lambda crops: [transforms.RandomResizedCrop(128)(crop) for crop in crops]),
    # transforms.Lambda(lambda crops: [transforms.RandomHorizontalFlip(p=0.5)(crop) for crop in crops]),
    # transforms.Lambda(lambda crops: [transforms.RandomRotation(degrees=60)(crop) for crop in crops]),
#     transforms.Lambda(lambda crops: [transforms.ColorJitter(brightness=[0.8,1.2], contrast=0.3, saturation=0.2, hue = 0.2)(crop) for crop in crops]),
#     transforms.Lambda(lambda crops: [transforms.ToTensor()(crop) for crop in crops]),
])

basic_tfm = transforms.Compose([
    transforms.Resize(128),
    transforms.RandomCrop(128),
    transforms.ToTensor()
])

aug_num = 10

## **Datasets**
The data is labelled by the name, so we load images and label while calling '__getitem__'

In [37]:
class FoodDataset(Dataset):

    def __init__(self,path,tfm=test_tfm,files = None):
        super(FoodDataset).__init__()
        self.path = path
        self.files = sorted([os.path.join(path,x) for x in os.listdir(path) if x.endswith(".jpg")])
        if files != None:
            self.files = files
        print(f"One {path} sample",self.files[0])
        self.transform = tfm
  
    def __len__(self):
        return len(self.files)
  
    def __getitem__(self,idx):
        
        fname = self.files[idx]
        im = Image.open(fname)
        
        if self.transform == test_tfm:
            imgs = [self.transform(im) for _ in range(aug_num - 1)]
            imgs.append(basic_tfm(im))
            imgs = torch.stack(imgs)
            imgs = imgs.view(3*aug_num, 128, 128)
        else:
            imgs = self.transform(im) 
        try:
            label = int(fname.split("/")[-1].split("_")[0])
        except:
            label = -1 # test has no label
        return imgs,label



In [38]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        # torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        # torch.nn.MaxPool2d(kernel_size, stride, padding)
        # input 維度 [3, 128, 128]
        self.cnn0 = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1),  # [32, 128, 128]
            nn.BatchNorm2d(32),
            nn.ReLU(), # [32, 128, 128]
#             nn.MaxPool2d(2, 2, 0),  
        )
        
        self.res0 = nn.Sequential(
            nn.Conv2d(32, 64, 1, 2, 0), #[64, 64, 64]
            nn.BatchNorm2d(64),
#             nn.ReLU(), #[64, 64, 64]
        )
        
        self.cnn1 = nn.Sequential(
            nn.Conv2d(32, 64, 3, 1, 1),  # [64, 128, 128]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),  # [64, 64, 64]
        )
        
        self.res1 = nn.Sequential(
            nn.Conv2d(64, 128, 1, 2, 0),
            nn.BatchNorm2d(128),
#             nn.ReLU(), #[128, 32, 32]
        )

        
        self.cnn2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, 1, 1), # [128, 64, 64]
            nn.BatchNorm2d(128),
#             nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [128, 32, 32]
        )
        
        self.res2 = nn.Sequential(
            nn.Conv2d(128, 256, 1, 2, 0),
            nn.BatchNorm2d(256),
#             nn.ReLU(), #[256,16,16]
        )        
        
        self.cnn3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, 1, 1), # [256, 32, 32]
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [256, 16, 16]
        )
        
        self.res3 = nn.Sequential(
            nn.Conv2d(256, 512, 1, 2, 0),
            nn.BatchNorm2d(512),
#             nn.ReLU(), #[512,8,8]
        )  
        
        self.cnn4 = nn.Sequential(
            nn.Conv2d(256, 512, 3, 1, 1), # [512, 16, 16]
            nn.BatchNorm2d(512),
#             nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),       # [512, 8, 8]
        )
        
        
        self.cnn5 = nn.Sequential(
            nn.Conv2d(512, 512, 3, 1, 1), # [512, 8, 8]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),       # [512, 4, 4]
        )
        
        self.relu = nn.ReLU()
        
        self.fc = nn.Sequential(
            
            nn.Linear(512*4*4,1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
             
            nn.Linear(256,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            
            nn.Linear(128, 11)
        )

    def forward(self, x):
        out = self.cnn0(x)
        
        res = self.res0(out)
        out = self.cnn1(out) + res
        out = self.relu(out)
        
        res = self.res1(out)
        out = self.cnn2(out) + res
        out = self.relu(out)
        
        res = self.res2(out)
        out = self.cnn3(out) + res
        out = self.relu(out)
        
        res = self.res3(out)
        out = self.cnn4(out) + res
        out = self.relu(out)
        
        out = self.cnn5(out)
        
        out = out.view(out.size()[0], -1)
        return self.fc(out)

In [9]:
_exp_name = "sample"
batch_size = 64
_dataset_dir = "food11"

# Construct datasets.
# The argument "loader" tells how torchvision reads the data.
train_set = FoodDataset(os.path.join(_dataset_dir,"training"), tfm = train_tfm)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  pin_memory=True)
valid_set = FoodDataset(os.path.join(_dataset_dir,"validation"), tfm = train_tfm)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True,  pin_memory=True)

One food11\training sample food11\training\0_0.jpg
One food11\validation sample food11\validation\0_0.jpg


In [ ]:
# "cuda" only when GPUs are available.
device = "cuda" if torch.cuda.is_available() else "cpu"

#debugging
CUDA_LAUNCH_BLOCKING=1

# The number of hyperparemeters
n_epochs = 200
patience = 300 # If no improvement in 'patience' epochs, early stop
lr = 0.0005
# Initialize a model, and put it on the device specified.
model = Classifier().to(device)

# For the classification task, we use cross-entropy as the measurement of performance.
criterion = nn.CrossEntropyLoss()

# Initialize optimizer, you may fine-tune some hyperparameters such as learning rate on your own.
optimizer = torch.optim.Adam(model.parameters(), lr = lr, weight_decay = 1e-5) 

# Initialize trackers, these are not parameters and should not be changed
stale = 0
best_acc = 0

for epoch in range(n_epochs):

    # ---------- Training ----------
    # Make sure the model is in train mode before training.
    model.train()

    # These are used to record information in training.
    train_loss = []
    train_accs = []

    for batch in tqdm(train_loader):

        # A batch consists of image data and corresponding labels.
        imgs, labels = batch
        
        #imgs = imgs.half()
        #print(imgs.shape,labels.shape)
        
        # Forward the data. (Make sure data and model are on the same device.)
        logits = model(imgs.to(device))
                       
        # Calculate the cross-entropy loss.
        # We don't need to apply softmax before computing cross-entropy as it is done automatically.
        loss = criterion(logits, labels.to(device))

        # Gradients stored in the parameters in the previous step should be cleared out first.
        optimizer.zero_grad()

        # Compute the gradients for parameters.
        loss.backward()

        # Clip the gradient norms for stable training.
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm= 1 ,norm_type = 2)

        # Update the parameters with computed gradients.
        optimizer.step()

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()

        # Record the loss and accuracy.
        train_loss.append(loss.item())
        train_accs.append(acc)
        
    train_loss = sum(train_loss) / len(train_loss)
    train_acc = sum(train_accs) / len(train_accs)

    # Print the information.
    print(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")

    # ---------- Validation ----------
    # Make sure the model is in eval mode so that some modules like dropout are disabled and work normally.
    model.eval()

    # These are used to record information in validation.
    valid_loss = []
    valid_accs = []

    # Iterate the validation set by batches.
    for batch in tqdm(valid_loader):
        
        # A batch consists of image data and corresponding labels.
        
        imgs, labels = batch
        
#         pre_len = len(imgs)
        
#         Im = torch.empty(pre_len*aug_num,3,128,128, dtype = torch.float32)
        
        
#         for i in range(pre_len):
#             Im[i*aug_num:(i+1)*aug_num,:,:,:] = imgs[i].view(aug_num,3,128,128)
        
#         imgs = Im
        
        #imgs = imgs.half()

        # We don't need gradient in validation.
                       
        # Using torch.no_grad() accelerates the forward process.
        with torch.no_grad():
            logits = model(imgs.to(device))
        
        
        # We can still compute the loss (but not the gradient).
#         loss = criterion(Logits, labels.to(device))
        
        # Compute the accuracy for current batch.
        acc = (torch.argmax(logits , dim =-1).to(device) == labels.to(device)).float().mean()
        
        

        # Record the loss and accuracy.
#         valid_loss.append(loss.item())
        valid_accs.append(acc)
        
        #break

    # The average loss and accuracy for entire validation set is the average of the recorded values.
#     valid_loss = sum(valid_loss) / len(valid_loss)
    valid_acc = sum(valid_accs) / len(valid_accs)

    # Print the information.
    print(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] acc = {valid_acc:.5f}")


    # update logs
    if valid_acc > best_acc:
        with open(f"./{_exp_name}_log.txt","a"):
            print(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] acc = {valid_acc:.5f} -> best")
            
    else:
        with open(f"./{_exp_name}_log.txt","a"):
            print(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] acc = {valid_acc:.5f}")


    # save models
    if valid_acc > best_acc:
        print(f"Best model found at epoch {epoch}, saving model")
        torch.save(model.state_dict(), f"{_exp_name}_best.ckpt") # only save best to prevent output memory exceed error
        best_acc = valid_acc
        stale = 0
    else:
        stale += 1
        if stale > patience:
            print(f"No improvment {patience} consecutive epochs, early stopping")
            break

In [39]:
test_set = FoodDataset(os.path.join(_dataset_dir,"test"), tfm = test_tfm)
test_loader = DataLoader(test_set, batch_size=batch_size//8, shuffle=False, num_workers=0, pin_memory=True)

One food11\test sample food11\test\0001.jpg


## Testing and generate prediction CSV

In [41]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_best = Classifier().to(device)
model_best.load_state_dict(torch.load(f"{_exp_name}_best.ckpt", map_location=torch.device('cpu') if device == 'cpu' else None))
model_best.eval()
prediction = []

with torch.no_grad():
    for data,_ in test_loader:

        pre_len = len(data)
        data = data.view(pre_len,aug_num,3,128,128)

        
        test_pred = torch.empty(pre_len, dtype = torch.int64)
        
        for i in range(pre_len):
            
            list = [0] * 11
            logits = model_best(data[i].to(device))
            for j in range(aug_num):
                if j == 0:
                    list[logits[j].argmax(dim = -1)] += 5
                else :
                    list[logits[j].argmax(dim = -1)] += 1
            mx = 0
            pos = 0
            
            for j in range(11):
                
                if mx < list[j]:
                    mx = list[j]
                    pos = j
            
            test_pred[i] = pos
            
        prediction += test_pred.tolist()

In [42]:
#create test csv
def pad4(i):
    return "0"*(4-len(str(i)))+str(i)
df = pd.DataFrame()
df["Id"] = [pad4(i) for i in range(1,len(test_set)+1)]
df["Category"] = prediction
df.to_csv("submission.csv",index = False)